# 09 — Sensitivity to PC2 and higher within-domain principal components

**Manuscript:** *Latent gait instability underlying retrospective fall occurrence in Parkinson’s disease*

## Aim

This notebook addresses whether the PC1-only reduction discards potentially relevant within-domain structure. It:

- reproduces the original domain definitions and preprocessing;
- reports all component variances and loadings for each domain;
- refits the higher-order latent construct after adding neuromotor-complexity PC2, all domain PC2s, and the components required to reach 80% cumulative variance per domain;
- quantifies changes in participant-level latent scores;
- repeats the Bayesian fall-occurrence association with identical covariates and latent-score uncertainty propagation;
- directly estimates the incremental association of neuromotor-complexity PC2 and, in a regularized joint model, all higher PCs.

The analysis is a prespecified sensitivity analysis. Higher components are not selected according to their outcome association.

In [ ]:
from __future__ import annotations

import json
import os
import platform
import warnings
from pathlib import Path

# Keep all library caches inside the writable project area. This matters in
# sandboxed/reproducible executions and has no effect on statistical results.
_launch_cwd = Path.cwd().resolve()
_cache_root = (_launch_cwd.parent if _launch_cwd.name == "notebooks" else _launch_cwd) / ".runtime_cache"
_cache_root.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("XDG_CACHE_HOME", str(_cache_root))
os.environ.setdefault("MPLCONFIGDIR", str(_cache_root / "matplotlib"))
os.environ.setdefault("PYTENSOR_FLAGS", f"base_compiledir={_cache_root / 'pytensor'}")

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc as pm
import pytensor.tensor as pt
import seaborn as sns
from IPython.display import Markdown, display
from scipy.special import ndtr
from scipy.stats import norm, spearmanr
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

VALIDATION_MODE = os.getenv("LATENT_GAIT_VALIDATION_MODE", "0") == "1"
QUICK_MODE = os.getenv("LATENT_GAIT_QUICK_MODE", "0") == "1"

# Publication defaults. Quick mode is only for code-path validation, never reporting.
DRAWS = int(os.getenv("LATENT_GAIT_DRAWS", "80" if QUICK_MODE else "2000"))
TUNE = int(os.getenv("LATENT_GAIT_TUNE", "80" if QUICK_MODE else "2000"))
CHAINS = int(os.getenv("LATENT_GAIT_CHAINS", "2" if QUICK_MODE else "4"))
CORES = int(os.getenv("LATENT_GAIT_CORES", str(min(CHAINS, 2))))
TARGET_ACCEPT = float(os.getenv("LATENT_GAIT_TARGET_ACCEPT", "0.92" if QUICK_MODE else "0.95"))

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
RESULTS_ROOT = PROJECT_ROOT / "results"
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})

print({
    "python": platform.python_version(),
    "pymc": pm.__version__,
    "arviz": az.__version__,
    "validation_mode": VALIDATION_MODE,
    "quick_mode": QUICK_MODE,
    "draws": DRAWS,
    "tune": TUNE,
    "chains": CHAINS,
})

## Provenance and required inputs

This notebook uses the exact raw column names and domain membership in notebooks 02 and 05 of the archived pipeline.

Required real-data input:

- `data/raw/DatasetIcotMond.xlsx`

Optional but strongly recommended audit input:

- `data/processed/latent_trunk_axes.csv`

The optional file is used only to verify/reconcile the arbitrary signs of the recomputed PC1 axes against the archived primary axes. PCA signs have no substantive meaning.

`LATENT_GAIT_VALIDATION_MODE=1` creates a synthetic schema-compatible fixture solely for testing execution. Synthetic estimates must never enter the manuscript or rebuttal.

In [ ]:
OUT_DIR = RESULTS_ROOT / ("_synthetic_validation/09_pca_components_processed" if VALIDATION_MODE else "09_pca_component_sensitivity_processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_PATH = PROJECT_ROOT / "data/raw/DatasetIcotMond.xlsx"
EXISTING_AXES_PATH = PROJECT_ROOT / "data/processed/latent_trunk_axes.csv"

DOMAINS = {
    "Rhythmicity-recurrence": [
        "HR V", "HR ML", "HR AP", "iHR V", "iHR ML", "iHR AP",
        "%det V", "%det ML", "%det AP",
    ],
    "Neuromotor complexity": ["MSE V", "MSE ML", "MSE AP"],
    "Lower trunk kinematics": ["Tilt", "Obliquity", "Rotation (range)"],
}
DOMAIN_SHORT = {
    "Rhythmicity-recurrence": "Rhythmic",
    "Neuromotor complexity": "Complexity",
    "Lower trunk kinematics": "Postural",
}

def synthetic_raw_fixture(n_subjects=269, seed=42):
    rg = np.random.default_rng(seed)
    ids = np.arange(1, n_subjects+1)
    latent = rg.normal(size=n_subjects)
    age = rg.normal(68, 8, n_subjects)
    duration = np.clip(rg.gamma(2.1, 3.1, n_subjects), .2, 27)
    complexity2 = rg.normal(size=n_subjects)
    eta = -0.7 + .38*latent + .16*complexity2 + .12*(age-age.mean())/age.std()
    y = rg.binomial(1, 1/(1+np.exp(-eta)))
    base = pd.DataFrame({"ID": ids, "Age": age, "Duration (years)": duration, "target_bin": y})
    patterns = {
        "Rhythmicity-recurrence": np.linspace(.85, .35, 9),
        "Neuromotor complexity": np.array([.85, .75, .7]),
        "Lower trunk kinematics": np.array([.9, .8, .65]),
    }
    for domain, cols in DOMAINS.items():
        load = patterns[domain]
        for j, col in enumerate(cols):
            signal = load[j]*latent
            if domain == "Neuromotor complexity":
                signal += np.array([.55, -.45, .15])[j]*complexity2
            base[col] = signal + rg.normal(0, .75, n_subjects)
    return base

if VALIDATION_MODE:
    raw = synthetic_raw_fixture()
    DATA_SOURCE = "synthetic schema-compatible fixture"
else:
    if not RAW_PATH.exists():
        raise FileNotFoundError(
            f"Missing essential input: {RAW_PATH}. The public code repository does not contain it, "
            "and Zenodo record 10.5281/zenodo.18095243 is restricted. Place the authorized workbook "
            "at data/raw/DatasetIcotMond.xlsx and rerun."
        )
    raw = pd.read_excel(RAW_PATH)
    DATA_SOURCE = str(RAW_PATH)

raw.columns = raw.columns.str.strip().str.replace("\n", " ", regex=False)
print("Data source:", DATA_SOURCE)
print("Rows:", len(raw), "Unique IDs:", raw["ID"].nunique())

## Reproduce preprocessing and derive the complete component set

To match notebook 02, all 15 trunk features are coerced to numeric, isolated missing observations are median-imputed, and each feature is standardized with population SD before separate within-domain PCA. The audit table makes this choice explicit because the manuscript text should use the same missing-data description as the executable pipeline.

PC signs are arbitrary. When the archived PC1 file is present, signs are aligned to it. Otherwise every component is oriented deterministically so its largest absolute loading is positive. All sensitivity variants use the same orientation.

In [ ]:
ALL_FEATURES = sum(DOMAINS.values(), [])
required_clin = ["ID", "Age", "Duration (years)", "target_bin"]
missing = [c for c in ALL_FEATURES + required_clin if c not in raw]
assert not missing, f"Missing required columns: {missing}"

X_raw = raw[ALL_FEATURES].apply(pd.to_numeric, errors="coerce")
missingness = X_raw.isna().mean().rename("missing_fraction").to_frame()
display(missingness.sort_values("missing_fraction", ascending=False))
missingness.to_csv(OUT_DIR / "raw_feature_missingness.csv")

imputer = SimpleImputer(strategy="median")
X_imp = pd.DataFrame(imputer.fit_transform(X_raw), columns=ALL_FEATURES, index=raw.index)
global_scaler = StandardScaler()
X_scaled = pd.DataFrame(global_scaler.fit_transform(X_imp), columns=ALL_FEATURES, index=raw.index)

existing_axes = pd.read_csv(EXISTING_AXES_PATH) if EXISTING_AXES_PATH.exists() else None
if existing_axes is not None:
    assert len(existing_axes) == len(raw), "Existing axis file is not row-aligned with the raw workbook."

score_df = pd.DataFrame(index=raw.index)
variance_rows, loading_rows = [], []
pca_objects = {}

for domain, features in DOMAINS.items():
    pca = PCA(n_components=len(features), random_state=RANDOM_SEED)
    scores = pca.fit_transform(X_scaled[features])
    components = pca.components_.copy()
    short = DOMAIN_SHORT[domain]

    for j in range(len(features)):
        sign = 1.0
        candidate_existing = f"PC{j+1}_{short}"
        if j == 0 and existing_axes is not None and candidate_existing in existing_axes:
            corr = np.corrcoef(scores[:, j], existing_axes[candidate_existing])[0, 1]
            sign = 1.0 if corr >= 0 else -1.0
        else:
            anchor = np.argmax(np.abs(components[j]))
            sign = 1.0 if components[j, anchor] >= 0 else -1.0
        scores[:, j] *= sign
        components[j] *= sign
        col = f"PC{j+1}_{short}"
        score_df[col] = scores[:, j]
        variance_rows.append({
            "domain": domain, "component": f"PC{j+1}",
            "explained_variance": pca.explained_variance_ratio_[j],
            "cumulative_variance": pca.explained_variance_ratio_[:j+1].sum(),
        })
        for feature, loading in zip(features, components[j]):
            loading_rows.append({"domain": domain, "component": f"PC{j+1}", "feature": feature, "loading": loading})
    pca_objects[domain] = pca

variance = pd.DataFrame(variance_rows)
loadings = pd.DataFrame(loading_rows)
display(variance)
variance.to_csv(OUT_DIR / "all_components_explained_variance.csv", index=False)
loadings.to_csv(OUT_DIR / "all_components_loadings.csv", index=False)
score_df.to_csv(OUT_DIR / "all_domain_component_scores_row_level.csv", index=False)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.7))
for ax, domain in zip(axes, DOMAINS):
    d = variance[variance.domain == domain]
    x = np.arange(1, len(d)+1)
    ax.bar(x, 100*d.explained_variance, color="#4C72B0", alpha=.75)
    ax.plot(x, 100*d.cumulative_variance, marker="o", color="black", label="cumulative")
    ax.axhline(80, color="#C44E52", ls="--", lw=1)
    ax.set(title=domain, xlabel="Principal component", ylabel="Variance explained (%)", xticks=x, ylim=(0, 105))
axes[-1].legend(frameon=False)
fig.tight_layout()
fig.savefig(OUT_DIR / "domain_scree_and_cumulative_variance.png", bbox_inches="tight")
plt.show()

## Prespecified sensitivity variants

- **Primary PC1-only:** the original one-axis-per-domain structure.
- **+ Complexity PC2:** the reviewer-targeted sensitivity.
- **+ all PC2:** PC2 from all three domains.
- **80% per domain:** the smallest number of within-domain PCs reaching at least 80% cumulative variance (without looking at falls).

The original latent measurement specification is retained: postural PC1 is the fixed loading anchor (loading 1, residual SD 0.1); all other loadings and residual SDs are estimated.

In [ ]:
def components_to_threshold(domain, threshold=.80):
    d = variance[variance.domain == domain].sort_values("component")
    return int(np.argmax(d.cumulative_variance.to_numpy() >= threshold) + 1)

primary = ["PC1_Postural", "PC1_Rhythmic", "PC1_Complexity"]
variants = {
    "Primary_PC1_only": primary,
    "Plus_complexity_PC2": primary + ["PC2_Complexity"],
    "Plus_all_PC2": primary + ["PC2_Postural", "PC2_Rhythmic", "PC2_Complexity"],
}
threshold_cols = []
for domain in DOMAINS:
    short = DOMAIN_SHORT[domain]
    n_keep = components_to_threshold(domain, .80)
    threshold_cols += [f"PC{j}_{short}" for j in range(1, n_keep+1)]
threshold_cols = ["PC1_Postural"] + [c for c in threshold_cols if c != "PC1_Postural"]
variants["Cumulative_80pct"] = threshold_cols

variant_table = pd.DataFrame([{"variant": k, "n_indicators": len(v), "indicators": ", ".join(v)} for k, v in variants.items()])
display(variant_table)
variant_table.to_csv(OUT_DIR / "latent_model_variants.csv", index=False)

## Fit the latent measurement models

Models are fit at the assessment-row level exactly as in notebook 05. Posterior mean and SD of the latent score are then aggregated by participant using the median, matching the original two-stage pipeline.

In [ ]:
raw_ids = raw["ID"].astype(str).str.strip()

def fit_latent_variant(name, indicators):
    indicators = ["PC1_Postural"] + [c for c in indicators if c != "PC1_Postural"]
    scaler = StandardScaler()
    Xz = scaler.fit_transform(score_df[indicators])
    n, k = Xz.shape
    coords = {"row": np.arange(n), "indicator": indicators, "free_indicator": indicators[1:]}
    with pm.Model(coords=coords) as model:
        severity = pm.Normal("severity", 0, 1, dims="row")
        loadings_free = pm.Normal("loadings_free", 0, 1, dims="free_indicator")
        sigma_free = pm.HalfNormal("sigma_free", .5, dims="free_indicator")
        loadings_all = pt.concatenate([pt.ones(1), loadings_free])
        sigma_all = pt.concatenate([pt.ones(1)*.1, sigma_free])
        mu = severity[:, None]*loadings_all[None, :]
        pm.Normal("X_obs", mu=mu, sigma=sigma_all, observed=Xz, dims=("row", "indicator"))
        idata = pm.sample(
            draws=DRAWS, tune=TUNE, chains=CHAINS, cores=CORES,
            target_accept=TARGET_ACCEPT, random_seed=RANDOM_SEED,
            return_inferencedata=True, progressbar=not QUICK_MODE,
        )
    sev_mean = np.asarray(idata.posterior["severity"].mean(("chain", "draw")))
    sev_sd = np.asarray(idata.posterior["severity"].std(("chain", "draw")))
    row_scores = pd.DataFrame({"ID": raw_ids, "latent_severity": sev_mean, "latent_severity_sd": sev_sd})
    subject = (row_scores.groupby("ID", as_index=False)
               .agg(latent_severity=("latent_severity", "median"),
                    latent_severity_sd=("latent_severity_sd", "median"),
                    n_obs=("latent_severity", "size")))
    idata.to_netcdf(OUT_DIR / f"latent_{name}.nc")
    return model, idata, subject

latent_models, latent_idatas, subject_scores = {}, {}, {}
for name, indicators in variants.items():
    print(f"\nFitting latent variant: {name}")
    latent_models[name], latent_idatas[name], subject_scores[name] = fit_latent_variant(name, indicators)
    subject_scores[name].to_csv(OUT_DIR / f"subject_scores_{name}.csv", index=False)

In [ ]:
latent_diag_rows = []
for name, idata in latent_idatas.items():
    sm = az.summary(idata, var_names=["loadings_free", "sigma_free"], hdi_prob=.95)
    latent_diag_rows.append({
        "stage": "latent_measurement",
        "model": name,
        "max_rhat": float(sm.r_hat.max()),
        "min_ess_bulk": float(sm.ess_bulk.min()),
        "min_ess_tail": float(sm.ess_tail.min()),
        "divergences": int(idata.sample_stats.diverging.sum()),
    })
latent_diagnostics = pd.DataFrame(latent_diag_rows)
display(latent_diagnostics)
latent_diagnostics.to_csv(OUT_DIR / "latent_sampling_diagnostics.csv", index=False)
if not QUICK_MODE and ((latent_diagnostics.max_rhat > 1.01).any() or (latent_diagnostics.divergences > 0).any()):
    warnings.warn("At least one latent measurement model needs sampling review before reporting.")

loading_rows = []
for name, idata in latent_idatas.items():
    arr = idata.posterior["loadings_free"].stack(sample=("chain", "draw"))
    for indicator in arr.free_indicator.values:
        x = np.asarray(arr.sel(free_indicator=indicator)).ravel()
        loading_rows.append({
            "variant": name, "indicator": str(indicator),
            "loading_mean": x.mean(),
            "loading_q025": np.quantile(x, .025),
            "loading_q975": np.quantile(x, .975),
            "P_loading_gt_0": (x > 0).mean(),
        })
latent_loading_summary = pd.DataFrame(loading_rows)
display(latent_loading_summary)
latent_loading_summary.to_csv(OUT_DIR / "latent_indicator_loading_summary.csv", index=False)

## How much does the latent construct change?

Each alternative score is sign-aligned to the primary score and standardized. We report Pearson and Spearman correlations, mean and 95th-percentile absolute score changes, and the percentage of participants moving to a different quartile. These are descriptive stability measures, not validation metrics.

In [ ]:
primary_scores = subject_scores["Primary_PC1_only"][["ID", "latent_severity"]].rename(columns={"latent_severity":"primary"})
construct_rows = []
aligned_scores = {"Primary_PC1_only": primary_scores.assign(score_z=lambda d: (d.primary-d.primary.mean())/d.primary.std(ddof=0))[["ID","score_z"]]}

for name, sdf in subject_scores.items():
    m = primary_scores.merge(sdf[["ID", "latent_severity"]], on="ID", validate="one_to_one")
    alt = m["latent_severity"].to_numpy()
    if np.corrcoef(m["primary"], alt)[0,1] < 0:
        alt = -alt
    pz = (m["primary"]-m["primary"].mean())/m["primary"].std(ddof=0)
    az_ = (alt-alt.mean())/alt.std(ddof=0)
    delta = az_ - pz
    q_primary = pd.qcut(pz, 4, labels=False, duplicates="drop")
    q_alt = pd.qcut(az_, 4, labels=False, duplicates="drop")
    construct_rows.append({
        "variant": name,
        "N": len(m),
        "pearson_r": float(np.corrcoef(pz, az_)[0,1]),
        "spearman_rho": float(spearmanr(pz, az_).statistic),
        "mean_abs_change_z": float(np.mean(np.abs(delta))),
        "p95_abs_change_z": float(np.quantile(np.abs(delta), .95)),
        "quartile_changed_fraction": float(np.mean(q_primary != q_alt)),
    })
    aligned_scores[name] = pd.DataFrame({"ID": m.ID, "score_z": az_})

construct_stability = pd.DataFrame(construct_rows)
display(construct_stability)
construct_stability.to_csv(OUT_DIR / "construct_stability_vs_primary.csv", index=False)

fig, axes = plt.subplots(1, len(variants)-1, figsize=(4*(len(variants)-1), 3.6), sharex=True, sharey=True)
for ax, name in zip(np.atleast_1d(axes), [n for n in variants if n != "Primary_PC1_only"]):
    m = aligned_scores["Primary_PC1_only"].merge(aligned_scores[name], on="ID", suffixes=("_primary", "_alt"))
    ax.scatter(m.score_z_primary, m.score_z_alt, s=15, alpha=.55)
    lim = [min(m.min(numeric_only=True)), max(m.max(numeric_only=True))]
    ax.plot(lim, lim, color="black", ls="--", lw=1)
    ax.set(title=name.replace("_", " "), xlabel="Primary latent score (z)", ylabel="Alternative latent score (z)")
fig.tight_layout()
fig.savefig(OUT_DIR / "latent_score_stability.png", bbox_inches="tight")
plt.show()

## Repeat the primary fall-occurrence association

Clinical data are aggregated exactly as in notebook 05: first age and disease duration, and the maximum binary fall indicator across repeated rows. One common complete-case participant set is locked for all variants. Each latent score is standardized and its posterior SD is converted to the same z scale. The Bayesian logistic model propagates uncertainty through a participant-specific `true_severity` term.

In [ ]:
clin = raw[["ID", "Age", "Duration (years)", "target_bin"]].copy()
clin["ID"] = clin["ID"].astype(str).str.strip()
for c in ["Age", "Duration (years)", "target_bin"]:
    clin[c] = pd.to_numeric(clin[c], errors="coerce")
clin_subject = (clin.groupby("ID", as_index=False)
                .agg(Age=("Age", "first"), Duration_years=("Duration (years)", "first"),
                     target_bin=("target_bin", "max")))
common_ids = set(clin_subject.dropna().ID)
for sdf in subject_scores.values():
    common_ids &= set(sdf.dropna().ID)
common_ids = sorted(common_ids)
assert len(common_ids) > 0
print("Locked common subject cohort:", len(common_ids))

def fit_association(name, score_df_subject):
    d = (clin_subject[clin_subject.ID.isin(common_ids)]
         .merge(score_df_subject, on="ID", validate="one_to_one")
         .sort_values("ID").reset_index(drop=True))
    sev_scale = d["latent_severity"].std(ddof=0)
    d["sev_z"] = (d["latent_severity"]-d["latent_severity"].mean())/sev_scale
    d["sev_sd_z"] = d["latent_severity_sd"]/sev_scale
    d["age_z"] = (d.Age-d.Age.mean())/d.Age.std(ddof=0)
    d["dur_z"] = (d.Duration_years-d.Duration_years.mean())/d.Duration_years.std(ddof=0)
    y = d.target_bin.astype(int).to_numpy()
    coords = {"subject": np.arange(len(d))}
    with pm.Model(coords=coords) as model:
        intercept = pm.Normal("intercept", 0, 1.5)
        beta_sev = pm.Normal("beta_sev", 0, 1)
        beta_age = pm.Normal("beta_age", 0, 1)
        beta_dur = pm.Normal("beta_dur", 0, 1)
        true_severity = pm.Normal("true_severity", d.sev_z.to_numpy(), d.sev_sd_z.to_numpy(), dims="subject")
        eta = intercept + beta_sev*true_severity + beta_age*d.age_z.to_numpy() + beta_dur*d.dur_z.to_numpy()
        p = pm.Deterministic("p_fall", pm.math.sigmoid(eta), dims="subject")
        pm.Bernoulli("falls", p=p, observed=y, dims="subject")
        idata = pm.sample(
            draws=DRAWS, tune=TUNE, chains=CHAINS, cores=CORES,
            target_accept=TARGET_ACCEPT, random_seed=RANDOM_SEED,
            return_inferencedata=True, idata_kwargs={"log_likelihood": True},
            progressbar=not QUICK_MODE,
        )
        pm.sample_posterior_predictive(idata, var_names=["falls", "p_fall"], random_seed=RANDOM_SEED,
                                       extend_inferencedata=True, progressbar=False)
    idata.to_netcdf(OUT_DIR / f"association_{name}.nc")
    return model, idata, d

assoc_models, assoc_idatas, assoc_data = {}, {}, {}
for name in variants:
    print(f"\nFitting association: {name}")
    assoc_models[name], assoc_idatas[name], assoc_data[name] = fit_association(name, subject_scores[name])

In [ ]:
assoc_rows = []
for name, idata in assoc_idatas.items():
    beta = np.asarray(idata.posterior["beta_sev"]).ravel()
    odds = np.exp(beta)
    loo = az.loo(idata, pointwise=True)
    sm = az.summary(idata, var_names=["beta_sev", "beta_age", "beta_dur"], hdi_prob=.95)
    assoc_rows.append({
        "variant": name,
        "N": len(assoc_data[name]),
        "events": int(assoc_data[name].target_bin.sum()),
        "beta_mean": beta.mean(),
        "OR_mean": odds.mean(),
        "OR_q025": np.quantile(odds, .025),
        "OR_q975": np.quantile(odds, .975),
        "P_beta_gt_0": (beta > 0).mean(),
        "max_rhat_reported": sm.r_hat.max(),
        "min_ess_bulk_reported": sm.ess_bulk.min(),
        "divergences": int(idata.sample_stats.diverging.sum()),
        "elpd_loo": float(loo.elpd_loo),
        "loo_se": float(loo.se),
        "pareto_k_gt_0.7": int((np.asarray(loo.pareto_k) > .7).sum()),
    })
association_summary = pd.DataFrame(assoc_rows)
primary_beta = association_summary.loc[association_summary.variant == "Primary_PC1_only", "beta_mean"].iloc[0]
association_summary["percent_change_log_OR_vs_primary"] = 100*(association_summary.beta_mean-primary_beta)/max(abs(primary_beta), 1e-9)
display(association_summary)
association_summary.to_csv(OUT_DIR / "fall_association_across_latent_variants.csv", index=False)

## Direct test of neuromotor-complexity PC2 and all higher PCs

Two additional models avoid conflating “does PC2 carry outcome information?” with “does PC2 belong in the single-factor measurement model?”

1. A targeted model adds neuromotor-complexity PC2 to the primary latent score.
2. A regularized joint model adds every PC2+ score simultaneously. The shared `Normal(0, 0.5)` prior limits unstable estimates and no component is selected after seeing the outcome.

These coefficients are exploratory associations, not causal effects and not evidence that a component should automatically be retained in the primary measurement model.

In [ ]:
component_subject = pd.concat([raw_ids.rename("ID"), score_df], axis=1)
component_subject = component_subject.groupby("ID", as_index=False).median(numeric_only=True)
higher_cols = [c for c in score_df.columns if not c.startswith("PC1_")]

base_d = assoc_data["Primary_PC1_only"].copy()
base_d = base_d.merge(component_subject[["ID"]+higher_cols], on="ID", validate="one_to_one")
for c in higher_cols:
    base_d[c] = (base_d[c]-base_d[c].mean())/base_d[c].std(ddof=0)

def fit_incremental(label, component_cols, prior_sd):
    d = base_d.copy()
    Xpc = d[component_cols].to_numpy()
    coords = {"subject": np.arange(len(d)), "component": component_cols}
    with pm.Model(coords=coords) as model:
        intercept = pm.Normal("intercept", 0, 1.5)
        beta_primary = pm.Normal("beta_primary", 0, 1)
        beta_age = pm.Normal("beta_age", 0, 1)
        beta_dur = pm.Normal("beta_dur", 0, 1)
        beta_component = pm.Normal("beta_component", 0, prior_sd, dims="component")
        true_primary = pm.Normal("true_primary", d.sev_z.to_numpy(), d.sev_sd_z.to_numpy(), dims="subject")
        eta = (intercept + beta_primary*true_primary + beta_age*d.age_z.to_numpy() + beta_dur*d.dur_z.to_numpy()
               + pm.math.dot(Xpc, beta_component))
        pm.Bernoulli("falls", logit_p=eta, observed=d.target_bin.astype(int).to_numpy(), dims="subject")
        idata = pm.sample(
            draws=DRAWS, tune=TUNE, chains=CHAINS, cores=CORES,
            target_accept=TARGET_ACCEPT, random_seed=RANDOM_SEED,
            return_inferencedata=True, idata_kwargs={"log_likelihood": True}, progressbar=not QUICK_MODE,
        )
    idata.to_netcdf(OUT_DIR / f"incremental_{label}.nc")
    return model, idata

inc_models, inc_idatas = {}, {}
inc_models["complexity_PC2"], inc_idatas["complexity_PC2"] = fit_incremental(
    "complexity_PC2", ["PC2_Complexity"], 1.0)
inc_models["all_higher_PCs"], inc_idatas["all_higher_PCs"] = fit_incremental(
    "all_higher_PCs", higher_cols, .5)

inc_rows = []
for label, idata in inc_idatas.items():
    arr = idata.posterior["beta_component"].stack(sample=("chain", "draw"))
    for comp in arr.component.values:
        b = np.asarray(arr.sel(component=comp)).ravel()
        o = np.exp(b)
        inc_rows.append({
            "model": label, "component": str(comp),
            "beta_mean": b.mean(), "OR_mean": o.mean(),
            "OR_q025": np.quantile(o, .025), "OR_q975": np.quantile(o, .975),
            "P_beta_gt_0": (b > 0).mean(),
        })
incremental_summary = pd.DataFrame(inc_rows)
display(incremental_summary)
incremental_summary.to_csv(OUT_DIR / "incremental_higher_PC_associations.csv", index=False)

incremental_diag_rows = []
for name, idata in inc_idatas.items():
    sm = az.summary(idata, var_names=["beta_primary", "beta_component", "beta_age", "beta_dur"], hdi_prob=.95)
    loo = az.loo(idata, pointwise=True)
    incremental_diag_rows.append({
        "stage": "incremental_association", "model": name,
        "max_rhat": float(sm.r_hat.max()), "min_ess_bulk": float(sm.ess_bulk.min()),
        "min_ess_tail": float(sm.ess_tail.min()),
        "divergences": int(idata.sample_stats.diverging.sum()),
        "pareto_k_gt_0.7": int((np.asarray(loo.pareto_k) > .7).sum()),
    })
incremental_diagnostics = pd.DataFrame(incremental_diag_rows)
display(incremental_diagnostics)
incremental_diagnostics.to_csv(OUT_DIR / "incremental_sampling_diagnostics.csv", index=False)

## Visual summary and posterior predictive checks

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.2))
plot_df = association_summary.iloc[::-1].reset_index(drop=True)
ypos = np.arange(len(plot_df))
ax.hlines(ypos, plot_df.OR_q025, plot_df.OR_q975, color="black", lw=1.5)
ax.plot(plot_df.OR_mean, ypos, "o", color="#4C72B0")
ax.axvline(1, color="#C44E52", ls="--")
ax.set(yticks=ypos, yticklabels=plot_df.variant.str.replace("_", " "), xlabel="Odds ratio per 1-SD latent score", title="Fall-occurrence association across PCA sensitivity variants")
fig.tight_layout()
fig.savefig(OUT_DIR / "association_forest_plot.png", bbox_inches="tight")
plt.show()

ppc_rows = []
for name, idata in assoc_idatas.items():
    yrep = np.asarray(idata.posterior_predictive["falls"]).reshape(-1, len(assoc_data[name]))
    prev = yrep.mean(axis=1)
    obs = assoc_data[name].target_bin.mean()
    ppc_rows.append({
        "variant": name, "observed_prevalence": obs, "replicated_mean": prev.mean(),
        "replicated_q025": np.quantile(prev,.025), "replicated_q975": np.quantile(prev,.975),
        "bayesian_p_value": (prev >= obs).mean(),
    })
ppc_summary = pd.DataFrame(ppc_rows)
display(ppc_summary)
ppc_summary.to_csv(OUT_DIR / "association_ppc_prevalence.csv", index=False)

## Reviewer-proof synthesis

The generated paragraph reports stability rather than declaring that a particular PC is “irrelevant.” A higher PC may describe orthogonal within-domain variation without loading on the same higher-order construct. Its direct outcome coefficient and its impact on the primary latent score answer complementary questions.

In [ ]:
primary_assoc = association_summary.set_index("variant").loc["Primary_PC1_only"]
alternatives = association_summary[association_summary.variant != "Primary_PC1_only"]
min_r = construct_stability[construct_stability.variant != "Primary_PC1_only"].pearson_r.min()
max_mad = construct_stability[construct_stability.variant != "Primary_PC1_only"].mean_abs_change_z.max()
max_beta_change = alternatives.percent_change_log_OR_vs_primary.abs().max()
pc2 = incremental_summary.query("model == 'complexity_PC2' and component == 'PC2_Complexity'").iloc[0]
pc2_loading = latent_loading_summary.query(
    "variant == 'Plus_complexity_PC2' and indicator == 'PC2_Complexity'"
).iloc[0]
joint = incremental_summary.query("model == 'all_higher_PCs'").copy()
joint_notable = joint[(joint.OR_q025 > 1) | (joint.OR_q975 < 1)]

if pc2.OR_q025 <= 1 <= pc2.OR_q975:
    pc2_sentence = "Its 95% credible interval included the null."
else:
    pc2_sentence = "Its 95% credible interval excluded the null in this exploratory internal analysis."

if len(joint_notable):
    details = "; ".join(
        f"{r.component}: OR={r.OR_mean:.2f} ({r.OR_q025:.2f} to {r.OR_q975:.2f})"
        for r in joint_notable.itertuples()
    )
    joint_sentence = (f"In the regularized joint model containing all higher PCs, {len(joint_notable)} of "
                      f"{len(joint)} component intervals excluded 1 ({details}); these exploratory multivariable "
                      "coefficients were not used for post hoc component selection.")
else:
    joint_sentence = ("In the regularized joint model containing all higher PCs, no component's 95% credible "
                      "interval excluded 1.")

rebuttal = f"""### Rebuttal-ready result: PCA component sensitivity

We re-ran the domain-specific PCA using the original feature definitions and preprocessing and examined all higher-order components without outcome-based selection. We then re-estimated the higher-order latent construct under three prespecified extensions: addition of neuromotor-complexity PC2, addition of PC2 from every domain, and retention of the minimum number of components required to explain at least 80% of variance within each domain.

Across these variants, correlations with the original PC1-only latent score were at least r={min_r:.4f}, and the largest mean absolute participant-level change was {max_mad:.3f} SD. The posterior log-odds coefficient for latent gait instability changed by at most {max_beta_change:.1f}% relative to the PC1-only analysis. Neuromotor-complexity PC2 showed little loading on the original higher-order factor (mean={pc2_loading.loading_mean:.2f}, 95% CrI {pc2_loading.loading_q025:.2f} to {pc2_loading.loading_q975:.2f}). In the targeted incremental fall-occurrence model, neuromotor-complexity PC2 had OR={pc2.OR_mean:.2f} per SD (95% CrI {pc2.OR_q025:.2f} to {pc2.OR_q975:.2f}; posterior P[β>0]={pc2.P_beta_gt_0:.3f}). {pc2_sentence} {joint_sentence}

These analyses test robustness rather than establish a unique component-retention rule. PC2 and higher components represent variance orthogonal to PC1 within each domain; limited loading on the common higher-order factor would not imply absence of biomechanical information, and any fall-occurrence association remains exploratory, internally estimated, and non-causal.

"""
if VALIDATION_MODE:
    rebuttal = "**SYNTHETIC VALIDATION OUTPUT — DO NOT REPORT.**\n\n" + rebuttal
display(Markdown(rebuttal))
(OUT_DIR / "rebuttal_ready_pca_sensitivity.md").write_text(rebuttal)

## Reporting checklist

Before using the results:

- run with the authorized raw workbook and publication sampling defaults;
- reconcile the manuscript's missing-data wording with the executable median-imputation step reproduced here;
- confirm participant and event counts against Results, Table 1, supplementary tables, and repository artifacts;
- inspect all R-hat, ESS, divergence, and Pareto-\(k\) diagnostics;
- present every component's explained variance and loadings, not only outcome-favorable components;
- label the PC2 association as exploratory and avoid outcome-driven redefinition of the primary construct;
- retain the absence of external validation as a prominent limitation.